# 🇹🇷 Turkish Stress Detection v2.0
## BERTurk + CRF + Weighted Loss

**Tahmini Süre**: GPU ile ~30-45 dakika

**Adımlar**:
1. Kütüphaneleri kur
2. Veri setini yükle
3. Modeli eğit
4. Sonuçları değerlendir

In [ ]:
# 1. Kütüphaneleri Kur
!pip install transformers torch pytorch-crf seqeval -q

import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Google Drive'ı Bağla (Veri için)
from google.colab import drive
drive.mount('/content/drive')

# Veri setini yükleyin:
# Drive'a Project-2/data/processed klasörünü yükleyin
# veya aşağıdaki demo veriyi kullanın

In [ ]:
# 3. Demo Veri Seti (Hızlı test için)
# Gerçek veri için bu hücreyi atlayın

import json
import os

# Demo veri oluştur
demo_samples = [
    {"words": ["Ali", "eve", "geldi", "."], "labels": [1, 0, 0, 0]},
    {"words": ["Yarın", "okula", "gideceğim", "."], "labels": [1, 0, 0, 0]},
    {"words": ["Ben", "kitabı", "okudum", "."], "labels": [1, 0, 0, 0]},
    {"words": ["Annem", "yemeği", "hazırladı", "."], "labels": [0, 1, 0, 0]},
    {"words": ["Dün", "markete", "gittim", "."], "labels": [1, 0, 0, 0]},
    {"words": ["Çocuklar", "parkta", "oynadı", "."], "labels": [0, 1, 0, 0]},
    {"words": ["Öğretmen", "dersi", "anlattı", "."], "labels": [1, 0, 0, 0]},
    {"words": ["Biz", "Ankara'ya", "gittik", "."], "labels": [0, 1, 0, 0]},
] * 500  # 4000 örnek

os.makedirs('data/processed', exist_ok=True)

# Split
train_data = demo_samples[:3500]
val_data = demo_samples[3500:3750]
test_data = demo_samples[3750:]

with open('data/processed/train.json', 'w') as f:
    json.dump(train_data, f)
with open('data/processed/val.json', 'w') as f:
    json.dump(val_data, f)
with open('data/processed/test.json', 'w') as f:
    json.dump(test_data, f)

print(f"Demo veri oluşturuldu: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

In [ ]:
# 4. Model ve Dataset Sınıfları

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizerFast, get_linear_schedule_with_warmup
from torchcrf import CRF
from tqdm import tqdm
import json
import os

# BERTurk + CRF Model
class BertCRF(nn.Module):
    def __init__(self, model_name="dbmdz/bert-base-turkish-cased", num_labels=3):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.num_labels = num_labels
        
        # CRF transition initialization
        with torch.no_grad():
            self.crf.transitions[0, 2] = -10.0  # O -> I forbidden
            self.crf.start_transitions[2] = -10.0  # Start with I forbidden
            self.crf.transitions[1, 2] = 2.0  # B -> I encouraged
    
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        emissions = self.classifier(self.dropout(outputs.last_hidden_state))
        
        result = {"logits": emissions}
        
        if labels is not None:
            labels_crf = labels.clone()
            labels_crf[labels == -100] = 0
            mask = attention_mask.bool()
            loss = -self.crf(emissions, labels_crf, mask=mask, reduction='mean')
            result["loss"] = loss
        
        return result
    
    def decode(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            emissions = self.classifier(outputs.last_hidden_state)
            mask = attention_mask.bool()
            return self.crf.decode(emissions, mask=mask)

# Dataset
class EmphasisDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=128):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        words = sample.get('words', sample.get('tokens', []))
        labels = sample.get('labels', sample.get('bio_tags', []))
        
        # Convert string labels if needed
        label2id = {"O": 0, "B-EMPHASIS": 1, "I-EMPHASIS": 2}
        if labels and isinstance(labels[0], str):
            labels = [label2id.get(l, 0) for l in labels]
        
        encoding = self.tokenizer(
            words, is_split_into_words=True, truncation=True,
            max_length=self.max_length, padding='max_length', return_tensors='pt'
        )
        
        # Align labels
        word_ids = encoding.word_ids()
        aligned_labels = []
        prev_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != prev_word_idx:
                aligned_labels.append(labels[word_idx] if word_idx < len(labels) else 0)
            else:
                aligned_labels.append(labels[word_idx] if word_idx < len(labels) else 0)
            prev_word_idx = word_idx
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(aligned_labels, dtype=torch.long)
        }

print("✓ Model ve Dataset sınıfları tanımlandı")

In [ ]:
# 5. Veri Yükleme

# Gerçek veri için bu yolu değiştirin:
# DATA_DIR = "/content/drive/MyDrive/Project-2/data/processed"
DATA_DIR = "data/processed"

def load_data(data_dir):
    datasets = {}
    for split in ['train', 'val', 'test']:
        path = os.path.join(data_dir, f'{split}.json')
        if os.path.exists(path):
            with open(path, 'r') as f:
                datasets[split] = json.load(f)
            print(f"  {split}: {len(datasets[split])} samples")
    return datasets

print("Loading data...")
datasets = load_data(DATA_DIR)

# Tokenizer
print("\nLoading tokenizer...")
tokenizer = BertTokenizerFast.from_pretrained("dbmdz/bert-base-turkish-cased")

# Datasets
train_dataset = EmphasisDataset(datasets['train'], tokenizer)
val_dataset = EmphasisDataset(datasets['val'], tokenizer)
test_dataset = EmphasisDataset(datasets['test'], tokenizer)

print(f"\n✓ Datasets created")

In [ ]:
# 6. Eğitim

# Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Model
print("\nLoading BERTurk + CRF model...")
model = BertCRF()
model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)

print(f"\n✓ Ready to train ({NUM_EPOCHS} epochs, {len(train_loader)} steps/epoch)")

In [ ]:
# 7. Training Loop

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, labels)
        loss = outputs['loss']
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    total_correct = 0
    total_tokens = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            
            predictions = model.decode(input_ids, attention_mask)
            
            for pred_seq, label_seq, mask in zip(predictions, labels, attention_mask):
                for p, l, m in zip(pred_seq, label_seq.tolist(), mask.tolist()):
                    if m == 1 and l != -100:
                        if p == l:
                            total_correct += 1
                        total_tokens += 1
    
    accuracy = total_correct / total_tokens if total_tokens > 0 else 0
    return accuracy

# Training
print("\n" + "="*50)
print("TRAINING STARTED")
print("="*50)

best_acc = 0
for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"Train Loss: {train_loss:.4f}")
    
    val_acc = evaluate(model, val_loader, device)
    print(f"Val Accuracy: {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  ✓ Saved best model (Acc: {best_acc:.4f})")

print(f"\n✓ Training complete! Best Val Acc: {best_acc:.4f}")

In [ ]:
# 8. Test Evaluation

print("\n" + "="*50)
print("TEST EVALUATION")
print("="*50)

# Load best model
model.load_state_dict(torch.load('best_model.pt'))
test_acc = evaluate(model, test_loader, device)

print(f"\n🎯 Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

In [ ]:
# 9. Inference Demo

def predict(model, tokenizer, text, device):
    model.eval()
    words = text.split()
    
    encoding = tokenizer(
        words, is_split_into_words=True, return_tensors='pt',
        padding=True, truncation=True, max_length=128
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    predictions = model.decode(input_ids, attention_mask)[0]
    
    id2label = {0: "O", 1: "B-EMPHASIS", 2: "I-EMPHASIS"}
    word_ids = encoding.word_ids()
    
    word_labels = []
    prev_word_idx = None
    for idx, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != prev_word_idx:
            if idx < len(predictions):
                word_labels.append(id2label[predictions[idx]])
        prev_word_idx = word_idx
    
    # Highlight
    highlighted = []
    for word, label in zip(words, word_labels):
        if label in ['B-EMPHASIS', 'I-EMPHASIS']:
            highlighted.append(f"**{word}**")
        else:
            highlighted.append(word)
    
    return words, word_labels, " ".join(highlighted)

# Test
test_sentences = [
    "Ali yarın okula gidecek.",
    "Ben kitabı okudum.",
    "Yarın Ankara'ya gideceğim."
]

print("\n📝 Inference Demo:")
for sent in test_sentences:
    words, labels, highlighted = predict(model, tokenizer, sent, device)
    print(f"\nInput: {sent}")
    print(f"Labels: {labels}")
    print(f"Output: {highlighted}")

In [ ]:
# 10. Model'i Kaydet

# Google Drive'a kaydet
import shutil
os.makedirs('/content/drive/MyDrive/Project-2/outputs', exist_ok=True)
shutil.copy('best_model.pt', '/content/drive/MyDrive/Project-2/outputs/best_model_v2.pt')
print("✓ Model saved to Google Drive")

# Sonuçları kaydet
results = {
    'best_val_accuracy': best_acc,
    'test_accuracy': test_acc,
    'epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'model': 'BERTurk + CRF'
}

with open('/content/drive/MyDrive/Project-2/outputs/results_v2.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n📊 Final Results:")
print(f"  Val Accuracy: {best_acc:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")